In [1]:
import pandas as pd

In [4]:
df=pd.read.csv  ( "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"],
    encoding="latin-1",
)


print(df.shape)
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))
print("Tekrar eden satır:", df.duplicated().sum())
df.head()

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
label
ham     0.866
spam    0.134
Name: proportion, dtype: float64
Tekrar eden satır: 403


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
from sklearn.model_selection import train_test_split

df = df.drop_duplicates().reset_index(drop=True)
print("Temizlenmiş boyut:", df.shape)

train_val, test = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42
)
train, val = train_test_split(
    train_val, test_size=0.1765, stratify=train_val["label"], random_state=42
)

for ad, parca in [("train", train), ("val", val), ("test", test)]:
    print(ad, parca.shape, parca["label"].value_counts(normalize=True).round(3).to_dict())

Temizlenmiş boyut: (5169, 2)
train (3617, 2) {'ham': 0.874, 'spam': 0.126}
val (776, 2) {'ham': 0.874, 'spam': 0.126}
test (776, 2) {'ham': 0.874, 'spam': 0.126}


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)

X_train = vectorizer.fit_transform(train["text"])
X_val = vectorizer.transform(val["text"])
X_test = vectorizer.transform(test["text"])

print("X_train boyutu:", X_train.shape)
print("Kelime sayısı:", len(vectorizer.get_feature_names_out()))
print(vectorizer.get_feature_names_out()[:20])

X_train boyutu: (3617, 9128)
Kelime sayısı: 9128
['00' '00 sub' '000' '000 bonus' '000 cash' '000 prize' '000 xmas'
 '008704050406' '008704050406 sp' '02' '02 06' '02 09' '0207' '0207 153'
 '03' '03 05' '03 this' '04' '05' '050703']


In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, train["label"])

val_pred = model.predict(X_val)
print(val_pred[:10])
print(val["label"].values[:10])

['ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']
['ham' 'spam' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

print("Accuracy :", round(accuracy_score(val["label"], val_pred), 4))
print("Precision:", round(precision_score(val["label"], val_pred, pos_label="spam"), 4))
print("Recall   :", round(recall_score(val["label"], val_pred, pos_label="spam"), 4))
print("F1       :", round(f1_score(val["label"], val_pred, pos_label="spam"), 4))
print()
print(confusion_matrix(val["label"], val_pred, labels=["ham", "spam"]))
print()
print(classification_report(val["label"], val_pred))

Accuracy : 0.9588
Precision: 1.0
Recall   : 0.6735
F1       : 0.8049

[[678   0]
 [ 32  66]]

              precision    recall  f1-score   support

         ham       0.95      1.00      0.98       678
        spam       1.00      0.67      0.80        98

    accuracy                           0.96       776
   macro avg       0.98      0.84      0.89       776
weighted avg       0.96      0.96      0.96       776



In [12]:
import numpy as np

kelimeler = np.array(vectorizer.get_feature_names_out())
agirliklar = model.coef_[0]
sira = np.argsort(agirliklar)

print("En çok HAM'a işaret edenler:", list(kelimeler[sira[:15]]))
print("En çok SPAM'e işaret edenler:", list(kelimeler[sira[-15:]]))

En çok HAM'a işaret edenler: ['my', 'me', 'that', 'it', 'ok', 'but', 'in', 'how', 'gt', 'lt', 'so', 'da', 'am', 'he', 'at']
En çok SPAM'e işaret edenler: ['com', '150p', 'from', 'uk', 'claim', 'mobile', 'www', 'reply', 'your', 'stop', 'text', 'to', 'free', 'txt', 'call']


In [14]:
from sklearn.metrics import recall_score, precision_score, f1_score

def raporla(ad, tahmin):
    p = precision_score(val["label"], tahmin, pos_label="spam")
    r = recall_score(val["label"], tahmin, pos_label="spam")
    f = f1_score(val["label"], tahmin, pos_label="spam")
    print(f"{ad:28s} precision={p:.3f} recall={r:.3f} f1={f:.3f}")

raporla("Basit model", val_pred)

model_w = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model_w.fit(X_train, train["label"])
raporla("class_weight='balanced'", model_w.predict(X_val))

model_c = LogisticRegression(max_iter=1000, C=10, random_state=42)
model_c.fit(X_train, train["label"])
raporla("C=10", model_c.predict(X_val))

Basit model                  precision=1.000 recall=0.673 f1=0.805
class_weight='balanced'      precision=0.917 recall=0.898 f1=0.907
C=10                         precision=1.000 recall=0.847 f1=0.917


In [15]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

test_pred = model_c.predict(X_test)

print(classification_report(test["label"], test_pred))
print(confusion_matrix(test["label"], test_pred, labels=["ham", "spam"]))

cikti = pd.DataFrame({
    "text": test["text"].values,
    "gercek": test["label"].values,
    "tahmin": test_pred,
    "spam_olasiligi": model_c.predict_proba(X_test)[:, 1].round(4),
})
cikti.to_csv("test_tahminleri.csv", index=False)

# Kaçırılan spam'lere bak (hata analizi)
kacan = cikti[(cikti["gercek"] == "spam") & (cikti["tahmin"] == "ham")]
print(len(kacan), "spam kaçtı")
kacan.head(10)

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       678
        spam       0.98      0.85      0.91        98

    accuracy                           0.98       776
   macro avg       0.98      0.92      0.95       776
weighted avg       0.98      0.98      0.98       776

[[676   2]
 [ 15  83]]
15 spam kaçtı


,text,gercek,tahmin,spam_olasiligi
33,Check Out Choose Your Babe Videos @ sms.shsex....,spam,ham,0.3435
35,"Free msg. Sorry, a service you ordered from 81...",spam,ham,0.3966
85,100 dating service cal;l 09064012103 box334sk38ch,spam,ham,0.2473
98,ringtoneking 84484,spam,ham,0.0226
106,"Latest News! Police station toilet stolen, cop...",spam,ham,0.0604
166,Bought one ringtone and now getting texts cost...,spam,ham,0.2725
243,As a registered optin subscriber ur draw 4 Â£1...,spam,ham,0.3487
341,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham,0.0897
467,Babe: U want me dont u baby! Im nasty and have...,spam,ham,0.0773
471,In The Simpsons Movie released in July 2007 na...,spam,ham,0.0152
